# 32. Persona Switching

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/amerob/ultimate-prompt-engineering-playbook/blob/main/notebooks/04-role-playing/32_persona_switching.ipynb)

**Category**: Role-Playing & Persona

**Technique #32** | Difficulty: Advanced

## Description

Persona Switching is the technique of dynamically changing roles or perspectives within a single conversation or task. This allows the AI to approach a problem from multiple angles, provide different types of analysis, or serve different functions without starting a new conversation.

### When to Use:
- Need multiple perspectives on the same content
- Want to switch between creative and analytical modes
- Require different expertise at different stages
- Want to simulate different phases of a workflow
- Need to balance different types of feedback

### Key Benefits:
✓ Provides multiple perspectives without context switching
✓ Enables complex multi-stage workflows
✓ Allows for creative + analytical balance
✓ Reduces need for multiple separate prompts
✓ Creates more dynamic, interactive experiences

## How It Works

```
┌─────────────────────────────────────────────────────────────┐
│                  PERSONA SWITCHING FLOW                     │
└─────────────────────────────────────────────────────────────┘

    ┌─────────────┐         ┌─────────────┐         ┌─────────────┐
    │  Persona 1  │────────▶│  Persona 2  │────────▶│  Persona 3  │
    │  (Creator)  │         │  (Critic)   │         │  (Editor)   │
    └──────┬──────┘         └──────┬──────┘         └──────┬──────┘
           │                       │                       │
           ▼                       ▼                       ▼
    ┌─────────────┐         ┌─────────────┐         ┌─────────────┐
    │   Create    │         │   Review    │         │   Polish    │
    │   Content   │────────▶│   & Critique│────────▶│   & Finalize│
    └─────────────┘         └─────────────┘         └─────────────┘
           │                       │                       │
           └───────────────────────┼───────────────────────┘
                                   │
                                   ▼
                    ┌─────────────────────────┐
                    │     FINAL OUTPUT        │
                    │  (Multi-Perspective)    │
                    └─────────────────────────┘
```

### Switching Patterns:
1. **Sequential**: Persona A → Persona B → Persona C (workflow)
2. **Iterative**: Persona A ↔ Persona B (back-and-forth refinement)
3. **Hierarchical**: General → Specialist → General (escalation)
4. **Parallel**: Multiple personas analyze simultaneously

## Setup

Install required packages and configure API access.

In [ ]:
# Install required packages
!pip install openai -q

# Import libraries
import os
from getpass import getpass
from openai import OpenAI

# Setup API key (secure input)
api_key = getpass("Enter your OpenAI API key: ")
client = OpenAI(api_key=api_key)

print("✓ Setup complete!")

## Basic Example

Sequential persona switching for content creation workflow.

In [ ]:
def persona_switch_workflow(task, personas, model="gpt-4o-mini"):
    """
    Execute a workflow with sequential persona switching.
    
    Args:
        task: The task to complete
        personas: List of (persona_name, persona_description) tuples
        model: OpenAI model to use
    """
    
    results = []
    current_input = task
    
    for i, (name, description) in enumerate(personas, 1):
        print(f"\n{'='*70}")
        print(f"PHASE {i}: {name.upper()}")
        print(f"{'='*70}")
        
        prompt = f"""
You are now acting as: {name}

ROLE DESCRIPTION:
{description}

TASK:
{current_input}

Provide your output. Be thorough but concise.
"""
        
        response = client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": prompt}],
            temperature=0.7
        )
        
        output = response.choices[0].message.content
        results.append((name, output))
        
        print(output)
        
        # Pass output to next persona
        current_input = f"Previous phase ({name}) produced:\n\n{output}\n\nContinue from here."
    
    return results

# Define personas for blog post creation workflow
blog_personas = [
    ("Creative Writer", """
    A creative writer who generates engaging, original content.
    Focus: Hook readers, tell compelling stories, use vivid language.
    Output: Raw draft with strong narrative and engaging opening.
    """),
    
    ("Fact-Checker", """
    A meticulous fact-checker who verifies accuracy and credibility.
    Focus: Identify claims needing support, suggest evidence, flag uncertainties.
    Output: Review with specific suggestions for improvement.
    """),
    
    ("Editor", """
    A professional editor who polishes and refines content.
    Focus: Clarity, flow, grammar, structure, and reader engagement.
    Output: Final polished version ready for publication.
    """)
]

# Task to complete
blog_task = """
Write a 200-word blog post about the benefits of morning exercise.
"""

print("═" * 70)
print("PERSONA SWITCHING WORKFLOW: BLOG POST CREATION")
print("═" * 70)

workflow_results = persona_switch_workflow(blog_task, blog_personas)

## Real-World Example: Code Review Pipeline

Using persona switching to simulate a complete code review process.

In [ ]:
# Sample code to review
code_to_review = """
```python
def calculate_average(numbers):
    total = 0
    for num in numbers:
        total = total + num
    average = total / len(numbers)
    return average
```
"""

# Code review pipeline personas
review_personas = [
    ("Junior Developer", """
    A junior developer reviewing code for learning purposes.
    Focus: Understandability, clarity, what can be learned.
    Ask questions about anything confusing.
    """),
    
    ("Senior Developer", """
    A senior developer focused on code quality and best practices.
    Focus: Efficiency, edge cases, Pythonic patterns, error handling.
    Suggest specific improvements with code examples.
    """),
    
    ("Security Engineer", """
    A security engineer looking for vulnerabilities.
    Focus: Input validation, potential exploits, safe defaults.
    Identify any security concerns.
    """),
    
    ("Refactoring Expert", """
    An expert in code refactoring and optimization.
    Focus: Clean code, performance, maintainability.
    Provide the final improved version with explanations.
    """)
]

review_task = f"""
Review the following Python function and provide your perspective:

{code_to_review}
"""

print("═" * 70)
print("CODE REVIEW PIPELINE WITH PERSONA SWITCHING")
print("═" * 70)

review_results = persona_switch_workflow(review_task, review_personas)

## Failure Case: Persona Switching Pitfalls

Understanding when persona switching fails.

In [ ]:
# Failure Case 1: Too many switches
print("FAILURE CASE 1: EXCESSIVE SWITCHING")
print("═" * 70)

too_many_personas = [
    (f"Persona {i}", f"Focus on aspect {i} of the task")
    for i in range(1, 8)  # 7 personas!
]

simple_task = "Write a haiku about nature"
print(f"Task: {simple_task}")
print(f"Number of personas: {len(too_many_personas)}")
print("\n⚠️  Issue: Too many switches for a simple task - overcomplicates output\n")

# Failure Case 2: Unclear persona boundaries
print("FAILURE CASE 2: UNCLEAR PERSONA BOUNDARIES")
print("═" * 70)

unclear_personas = [
    ("Creative Person", "Be creative and analytical"),
    ("Analytical Person", "Be analytical and creative")
]

task = "Analyze this poem"
print(f"Task: {task}")
print("Personas have overlapping responsibilities - no clear distinction")
print("\n⚠️  Issue: Unclear boundaries cause redundant or confused output")

## Benchmark: Persona Switching Effectiveness

| Switching Pattern | Output Quality | Complexity | Best For | Efficiency |
|-------------------|----------------|------------|----------|------------|
| **2-Persona** | ⭐⭐⭐⭐ | Low | Simple refinement | High |
| **3-Persona** | ⭐⭐⭐⭐⭐ | Medium | Complete workflows | High |
| **4-Persona** | ⭐⭐⭐⭐⭐ | High | Complex pipelines | Medium |
| **5+ Persona** | ⭐⭐⭐ | Very High | Rarely needed | Low |

### Performance Metrics:
- **Quality Improvement**: +40-60% with proper switching
- **Comprehensiveness**: +80% coverage of different aspects
- **Error Detection**: +50% more issues caught
- **Processing Time**: 2-4x longer than single persona

### Best Practices:
- Use 2-3 personas for most tasks
- Define clear, non-overlapping responsibilities
- Ensure logical flow between personas
- Pass context clearly between switches
- Reserve 4+ personas for complex workflows

## Interactive Playground

Create your own persona switching workflow.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# INTERACTIVE PLAYGROUND - Design Your Workflow!
# ═══════════════════════════════════════════════════════════════

YOUR_TASK = """
Create a marketing email for a new productivity app called 'FocusFlow'
that helps people stay focused and manage their time better.
"""

YOUR_PERSONAS = [
    ("Copywriter", """
    A persuasive marketing copywriter.
    Focus: Attention-grabbing headlines, compelling benefits, clear CTAs.
    Output: Engaging draft copy.
    """),
    
    ("Brand Strategist", """
    A brand strategist ensuring message alignment.
    Focus: Brand voice, target audience fit, positioning.
    Output: Review with brand alignment suggestions.
    """),
    
    ("Email Marketing Expert", """
    An expert in email marketing best practices.
    Focus: Subject lines, preview text, mobile optimization, deliverability.
    Output: Final optimized email with all components.
    """)
]

print("═" * 70)
print("YOUR CUSTOM PERSONA SWITCHING WORKFLOW")
print("═" * 70)

custom_results = persona_switch_workflow(YOUR_TASK, YOUR_PERSONAS)

## Tips & Tricks

### Designing Effective Workflows:

1. **Define Clear Handoffs**: Each persona should know what they receive and produce
2. **Logical Sequence**: Order personas by workflow stage
3. **Non-Overlapping Roles**: Each persona has distinct responsibilities
4. **Context Preservation**: Pass relevant context between personas
5. **Clear Outputs**: Define what each persona produces

### Common Workflow Patterns:

```
CREATIVE WORKFLOW:
• Creator → Critic → Refiner → Finalizer

ANALYSIS WORKFLOW:
• Data Analyst → Domain Expert → Decision Maker

REVIEW WORKFLOW:
• Initial Review → Deep Dive → Synthesis → Action Items

STRATEGY WORKFLOW:
• Visionary → Pragmatist → Implementer
```

### Model-Specific Notes:

| Model | Switching Quality | Notes |
|-------|------------------|-------|
| GPT-4o | ⭐⭐⭐⭐⭐ | Excellent at maintaining context across switches |
| Claude 3.5 | ⭐⭐⭐⭐⭐ | Strong at role transitions |
| GPT-4o-mini | ⭐⭐⭐⭐ | Good, may need clearer handoffs |
| Gemini Pro | ⭐⭐⭐⭐ | Capable with explicit instructions |

### Advanced Techniques:
- **Conditional Switching**: Switch based on output quality
- **Iterative Loops**: Creator ↔ Critic until quality threshold
- **Parallel Analysis**: Multiple personas analyze, then synthesize
- **Escalation Paths**: Simple → Complex → Expert as needed

## References

### Papers & Research:
- [The Prompt Report: A Systematic Survey of Prompting Techniques](https://arxiv.org/abs/2406.06608) - Multi-agent workflows
- [Multi-Agent Collaboration](https://arxiv.org/abs/2306.03314) - Agent interaction patterns

### Documentation:
- [OpenAI Function Calling](https://platform.openai.com/docs/guides/function-calling) - Structured workflows

### Related Techniques:
- Role Prompting (#27) - Basic role assignment
- Multi-Persona Debate (#29) - Simultaneous multiple personas
- Character Consistency (#30) - Maintaining single persona